## Boilerplate

In [1]:
import os
import numpy as np
import pandas as pd
import networkx as nx
import matplotlib.pyplot as plt
import pprint
import mygene
import json
from pathlib import Path

while "src" not in os.listdir():
    os.chdir('..')


from src.reading_in import read_in_proteins

FILEPATH = "data/raw/protein.links.v12.0.txt.gz"
G = read_in_proteins(include_essential_proteins=False, connected=True)


## Getting Expression Data

In [ ]:
file_path = "final_code_submission/Exclusion/ppi-data/expression/GSE3431_series_matrix.txt"

# Find the start and end of the expression table
with open(file_path, "r") as f:
    lines = f.readlines()

start = next(
    i for i, line in enumerate(lines)
    if "!series_matrix_table_begin" in line
)
end = next(
    i for i, line in enumerate(lines)
    if "!series_matrix_table_end" in line
)

# Read only the expression table
expression_df = pd.read_csv(
    file_path,
    sep="\t",
    skiprows=start + 1,
    nrows=end - start - 2,
    index_col=0
)

print(expression_df.shape)


(9335, 36)


## Relabelling Expression Data

In [ ]:


soft_path = "final_code_submission/Exclusion/ppi-data/expression/GPL90_family.soft"

# Locate the platform's probe annotation table.
with open(soft_path, encoding="utf-8", errors="replace") as f:
    lines = f.readlines()

begin = next(
    i for i, line in enumerate(lines)
    if line.startswith("!platform_table_begin")
)
end = next(
    i for i, line in enumerate(lines)
    if line.startswith("!platform_table_end")
)

annotation_df = pd.read_csv(
    soft_path,
    sep="\t",
    skiprows=begin + 1,
    nrows=end - begin - 1,
    dtype=str
)

# Find the relevant columns without depending on their position.
probe_col = next(
    c for c in annotation_df.columns
    if c.strip().lower() == "id"
)
gene_col = next(
    c for c in annotation_df.columns
    if c.strip().lower() == "orf"
)


# Map Affymetrix probes to yeast systematic gene IDs
probe_map = annotation_df[[probe_col, gene_col]].copy()
probe_map.columns = ["probe_id", "gene_id"]

probe_map["probe_id"] = probe_map["probe_id"].astype(str).str.strip()
probe_map["gene_id"] = probe_map["gene_id"].astype(str).str.strip()

# Nuclear genes, including IDs with suffixes such as YAL016C-B,
# and mitochondrial genes such as Q0010.
valid_gene_pattern = r"^(?:Y[A-P][LR][0-9]{3}[CW](?:-[A-Z])?|Q[0-9]{4})$"

probe_map = probe_map[
    probe_map["gene_id"].str.fullmatch(valid_gene_pattern, na=False)
].copy()

# Map expression probes to gene IDs
expr = expression_df.copy()
expr.index.name = "probe_id"
expr = expr.reset_index()

expr["probe_id"] = expr["probe_id"].astype(str).str.strip()
expr = expr.merge(probe_map, on="probe_id", how="inner")

# Average probes mapping to the same gene
gene_expression = (
    expr.groupby("gene_id")[expression_df.columns.tolist()]
    .mean()
)

gene_expression.to_csv("final_code_submission/Exclusion/GSE3431_gene_expression.csv")
probe_map.to_csv("final_code_submission/Exclusion/GSE3431_GPL90_probe_mapping.csv", index=False)

print("Mapped probe sets:", expr["probe_id"].nunique())
print("Distinct mapped genes:", gene_expression.shape[0])
print("Graph nodes:", len(G.nodes()))
print("Matched graph nodes:",
      len(set(map(str, G.nodes())) & set(gene_expression.index)))


graph_nodes = set(map(str, G.nodes()))
matched = set(gene_expression.index) & graph_nodes

print("Graph nodes:", len(graph_nodes))
print("Genes with expression data:", len(matched))
print("Fraction of graph covered:",
      round(len(matched) / len(graph_nodes), 3))
# Checking how much is mapped
print("Mapped probes:", expr["probe_id"].nunique())
print("Distinct genes:", gene_expression.shape[0])
print("Expression matrix shape:", gene_expression.shape)



Mapped probe sets: 6296
Distinct mapped genes: 6088
Graph nodes: 4209
Matched graph nodes: 3948
Graph nodes: 4209
Genes with expression data: 3948
Fraction of graph covered: 0.938
Mapped probes: 6296
Distinct genes: 6088
Expression matrix shape: (6088, 36)
